# The GHZ benchmark

A GHZ state spreads one superposition across every qubit at once, so its fidelity is a blunt, honest
measure of how much entanglement a device can actually hold. A fidelity above **0.5** is a sufficient
condition for the state to be *genuinely multipartite entangled* (Leibfried et al., Nature 438, 639
(2005)) - so that line is the one worth crossing.

``iqm-benchmarks`` ships this as a ready-made benchmark. This notebook is a tutorial on the package: how a
benchmark is configured, run and analysed, using GHZ as the worked example. It sweeps GHZ sizes from 5 to
20 qubits in a single run, and then shows that **how you prepare the state is itself an error reduction
technique** - the same qubits, the same shots, a different circuit, and a measurably different fidelity.

1. **Setup** - installing ``iqm-benchmarks``, imports, token, backend.
2. **How iqm-benchmarks is put together** - the configure, run, analyse pattern shared by every benchmark.
3. **Look at the chip first** - the connectivity and fidelity graph.
4. **The configuration** - what each ``GHZConfiguration`` field does, and how the fidelity is measured.
5. **Running and analysing** - one run, sixteen layouts, and where the device falls below 0.5.
6. **Why the tree circuit** - routing the CZs along the best gates, and in fewer layers.
7. **The same lesson on a star QPU** - chain vs fan-out, and the gate counts behind the difference.

## 1. Setup

Everything in this notebook comes from **``iqm-benchmarks``**, which is on public PyPI:

```
pip install iqm-benchmarks
```

That one package is enough. It declares ``iqm-client[qiskit]`` as a dependency, so the provider, the
backend and the Qiskit integration used below are installed along with it - there is no second package to
remember.

Run the cell below once per environment; re-running it is harmless if the package is already there.

In [ ]:
%pip install --quiet iqm-benchmarks iqm-qubit-selector matplotlib


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
import os

from iqm.benchmarks.entanglement.ghz import GHZBenchmark, GHZConfiguration
from iqm.benchmarks.utils_plots import plot_layout_fidelity_graph
from iqm.qiskit_iqm import IQMProvider

## Plotting helpers for this demo, defined next to this notebook.
from utils import apply_demo_style

apply_demo_style()  ## also restyles the plots that iqm-benchmarks produces itself

The token is read from the ``IQM_TOKEN`` environment variable. Setting it inside the notebook, as below, is
convenient but **not recommended**: it is easy to expose the token accidentally, for example when giving a
presentation or when committing the notebook to a shared repository. Prefer exporting it in your shell
before starting Jupyter.

In [20]:
import os

from dotenv import load_dotenv

load_dotenv()  # loads IQM_TOKEN from a local .env file into os.environ
token = os.getenv("IQM_TOKEN")
quantum_computer = "garnet" 
iqm_server_url = "https://resonance.iqm.tech/"  # provide your actual IQM server URL
os.environ["IQM_SERVER_URL"] = iqm_server_url
provider = IQMProvider(iqm_server_url, quantum_computer=quantum_computer)
backend = provider.get_backend()
print(f"Connected to {quantum_computer} with {backend.num_qubits} qubits.")

Connected to garnet with 20 qubits.


## 2. How iqm-benchmarks is put together

Every benchmark in the package follows the same three-step shape, so learning it once covers quantum
volume, randomized benchmarking, GST and the rest:

1. **Configure.** A ``...Configuration`` object - a pydantic model - holds every knob: which qubits, how
   many shots, which estimator, which mitigations. It is data, so you can build several, diff them, and
   keep them next to your results.
2. **Run.** ``Benchmark(backend, configuration).run()`` builds the circuits, transpiles them to the device
   and submits them. It returns a ``BenchmarkRunResult``: an ``xarray`` dataset of counts plus the
   circuits that produced them.
3. **Analyse.** ``.analyze()`` turns those counts into a ``BenchmarkAnalysisResult`` with two useful
   attributes:
   - ``observations`` - a list of ``BenchmarkObservation(name, value, uncertainty, identifier)``, where
     the identifier says which qubit layout the number belongs to,
   - ``plots`` - a dict of ready-made figures, shown all at once with ``result.plot_all()``.

The split matters in practice: ``run`` costs hardware time, ``analyze`` does not. You can re-analyse a run
as often as you like without touching the device again.

## 3. Look at the chip first

Before choosing anything, look at what the device offers today. ``plot_layout_fidelity_graph`` draws the
connectivity with the current calibration baked into the geometry:

- **Edge width** is the CZ error, as $w_{ij} = -\log(\mathcal{F}^{ij}_{\mathrm{CZ}})$, so **thinner is
  better**. Each edge carries its value as a label.
- **Node size** is the single-qubit error on the same log scale, so **smaller is better**. ``sq_metric``
  chooses which one: ``"fidelity"`` for gate fidelity, ``"readout"`` for readout, ``"coherence"`` for the
  idle fidelity.
- Qubits passed in ``qubit_layouts`` are marked in orange.
- ``show_ghz_path=True`` highlights in red the edges the GHZ tree circuit would actually use - useful here,
  because those are the CZ fidelities that decide the result.

This notebook does not run one GHZ state but a **whole range of sizes**, from 5 qubits up to 20. That turns
a single number into a curve, and the interesting question becomes not "what is the fidelity" but "how far
up can this device go before the state stops being genuinely entangled". The graph below marks the largest
of those layouts.

In [ ]:
min_qubits, max_qubits = 5, 20

## One layout per size: the first n qubits of the chip, for every n in the range.
layouts = [list(range(n)) for n in range(min_qubits, max_qubits + 1)]

print(f"{len(layouts)} layouts, from {min_qubits} to {max_qubits} qubits")

fig = plot_layout_fidelity_graph(backend, qubit_layouts=[layouts[-1]], sq_metric="coherence", show_ghz_path=True)

## 4. The configuration

``GHZConfiguration`` is a pydantic model that collects everything the benchmark needs to know. Nothing runs
when you build it - it is just data, which is why you can create several, diff them, and store one next to
the results it produced.

| Field | What it does |
|---|---|
| ``custom_qubits_array`` | the qubit layouts to run on, as lists of qiskit indices - one benchmark run covers all of them |
| ``shots`` | shots per circuit |
| ``state_generation_routine`` | how the GHZ circuit is built - ``"naive"``, ``"log"`` or ``"tree"`` on a crystal, ``"star"`` on a star device. Section 7 measures what this choice is worth |
| ``fidelity_routine`` | how the fidelity is estimated - ``"coherences"`` throughout this notebook |
| ``rem`` | whether readout error mitigation is applied |
| ``mit_shots`` | shots spent calibrating that mitigation |
| ``use_dd`` / ``dd_strategy`` | whether to apply dynamical decoupling, and which strategy |
| ``max_circuits_per_batch`` | how many circuits go to the device per submission |
| ``qiskit_optim_level`` / ``optimize_sqg`` | transpilation effort, and single-qubit gate squashing |

### How the fidelity is actually measured

You cannot read a state's fidelity off the device directly - the benchmark has to reconstruct it from many
measurements of many copies of the state. ``fidelity_routine="coherences"``, used throughout this notebook,
does that with **multiple quantum coherences**.

It prepares the GHZ state, applies the same phase $\phi$ to every qubit, then un-prepares the state and
measures how much amplitude returns to $|0\dots0\rangle$. Sweeping $\phi$ makes that probability
oscillate, and the amplitude of the oscillation at frequency $n$ gives the coherence between
$|0\dots0\rangle$ and $|1\dots1\rangle$ - the off-diagonal half of the fidelity. The populations, the
diagonal half, come from one extra circuit that simply measures the state.

Two practical consequences worth knowing before the run:

- The sweep is a fixed $2n + 2$ phase points for an $n$-qubit state, so the number of circuits is set
  entirely by the qubit count - there is no knob to turn. With a layout per size from 5 to 20, that cost is
  paid once per layout, which is why the cell below prints the total before submitting anything.
- The estimate comes from that single sweep, so it arrives **without an error bar**:
  ``observation.uncertainty`` is ``None``.

### This notebook's configuration

``CONFIG`` below fixes everything except the size: the tree circuit, no mitigation of any kind, and the
whole range of layouts in ``custom_qubits_array`` so a single run produces the full curve.
[4_ghz_error_reduction.ipynb](4_ghz_error_reduction.ipynb) picks it up from here and adds the techniques
that cost something, one at a time.

In [ ]:
shots = 1000

CONFIG = GHZConfiguration(
    state_generation_routine="tree",  ## see section 6 for why this is the right default on a crystal
    custom_qubits_array=layouts,  ## every size from 5 to 20, all in one run
    shots=shots,
    fidelity_routine="coherences",  ## the phase sweep described above
    rem=False,  ## no readout error mitigation
    use_dd=False,  ## no dynamical decoupling
    max_circuits_per_batch=100,
)

## Each layout costs its own 2n+2 phase circuits plus one for the populations.
print(f"Circuits to be submitted: {sum(2 * len(layout) + 3 for layout in layouts)}")

## 5. Running and analysing

``run()`` submits the circuits; ``analyze()`` turns the counts into observations and plots. Pulling one
number out of a result is a two-line filter, so it is worth defining once - the error reduction notebook
reuses exactly this helper.

In [23]:
def ghz_fidelity(result, qubit_layout, name="fidelity"):
    """Pick one fidelity observation of one layout out of an analysis result."""
    for observation in result.observations:
        if observation.identifier.string_identifier == str(list(qubit_layout)) and observation.name == name:
            return observation.value
    raise KeyError(f"No {name} observation for layout {qubit_layout}")

In [ ]:
benchmark = GHZBenchmark(backend, CONFIG)
run = benchmark.run()  ## builds, transpiles and submits the circuits
result = benchmark.analyze()  ## counts -> observations and plots

fidelities = {len(layout): ghz_fidelity(result, layout) for layout in layouts}

print("qubits  fidelity")
for n, value in fidelities.items():
    flag = "" if value > 0.5 else "   <- below the entanglement threshold"
    print(f"{n:>6}  {value:.4f}{flag}")

### Reading the results

``result.observations`` is a flat list covering **every layout in the configuration**, which is why one run
gave sixteen fidelities. Filter it by the layout's string identifier to get one layout's numbers. As noted
above, the coherences routine reports no uncertainty estimate, so ``observation.uncertainty`` is ``None``.

``result.plot_all()`` shows the figures the analysis produced. In the fidelity plot the labels ``L0``,
``L1``, ... enumerate the layouts in the order the configuration listed them - here, in order of increasing
size.

In [ ]:
## Every observation of the largest layout, to show what an observation carries.
for observation in result.observations:
    if observation.identifier.string_identifier == str(layouts[-1]):
        print(f"{observation.name}: {observation.value} +/- {observation.uncertainty}")

result.plot_all()

In [ ]:
## The transpiled circuit for the largest layout, as the device ran it.
circuit = run.circuits["transpiled_circuits"][f"{layouts[-1]}_native_ghz"].circuits[0]
circuit.draw("mpl", fold=True, idle_wires=False)

## 6. Why the tree circuit

The run above used ``state_generation_routine="tree"``. It is worth saying what that actually does, because
it is doing two independent things at once and both of them matter.

**It routes the CZs optimally.** The routine takes the connectivity restricted to your layout, weights
every edge by $-\log(\mathcal{F}_{\mathrm{CZ}})$ - the error of that two-qubit gate on today's calibration
- and computes a **minimum spanning tree**. The tree that comes out is the cheapest way to touch every
qubit in the layout, so the entanglement is carried by the best gates the chip currently has, rather than
by whichever gates the qubit index order happened to line up.

**It parallelises the CZs.** The routine then picks the most *central* qubit of that tree - the one whose
largest distance to any other qubit is smallest - and grows the state outward from it, shell by shell.
Every qubit that is already entangled can entangle a new neighbour in the same layer, so the CZ count stays
at $n-1$ (you cannot do better) but those gates pack into far fewer **layers**: depth logarithmic in $n$
instead of linear. Less depth means less time for the state to decohere while the circuit runs.

The contrast is with ``"naive"``, the textbook chain - one Hadamard, then one CX per qubit, handed down the
line. It is blind on both counts: linear depth, and no idea which gates are good.

Because the run covered every size from 5 to 20, the depth growth is visible directly in the circuits it
built. A linear construction would add roughly one layer per extra qubit; the tree should grow far more
slowly than that.

In [ ]:
## The untranspiled GHZ circuits the tree routine built, one per layout size.
untranspiled = run.circuits["untranspiled_circuits"]

print("qubits  depth  2q gates")
for layout in layouts:
    ghz = untranspiled[f"{layout}_native_ghz"].circuits[0]
    two_qubit = sum(count for name, count in ghz.count_ops().items() if name in ("cx", "cz"))
    print(f"{len(layout):>6}  {ghz.depth():>5}  {two_qubit:>8}")

## 7. The same lesson on a star QPU

A star device has no qubit-qubit couplers at all. Every qubit talks only to a central **computational
resonator**, and a two-qubit gate is a `MOVE` of one qubit's state into the resonator, a CZ against it, and
a `MOVE` back. That changes which circuit is cheap, so the routine that wins on a crystal is not the one
that wins here - which is the point: state preparation has to match the topology.

| Routine | What it builds | Why it costs what it costs |
|---|---|---|
| ``"naive"`` | the textbook linear chain | nothing in the chain is native. Each link entangles qubit $i$ with qubit $i+1$, and since no two qubits are directly coupled, every one of those links has to be shuttled through the resonator |
| ``"star"`` | a fan-out: one H, then a CX from that qubit to every other | matches the topology exactly - one qubit is the hub, just like the resonator - so the state can be parked in the resonator and every other qubit entangled against it |

The chain is the wrong shape for this chip and the fan-out is the right one. Each runs in its own cell
below, and the cell after them pulls out the **transpiled** circuits - the ones the device actually
executes, after `MOVE` routing - and puts their gate counts side by side.

In [34]:
star_computer = "sirius"  ## replace with your star device
star_nqubits = 8

star_backend = IQMProvider(iqm_server_url, quantum_computer=star_computer).get_backend()
star_layout = list(range(star_nqubits))

print(f"Connected to {star_computer}: {star_backend.num_qubits} qubits")
print(f"Computational resonators: {star_backend.architecture.computational_resonators}")

Connected to sirius: 16 qubits
Computational resonators: ['COMPR1']


In [ ]:
## The chain, on a chip that has no chains in it.
STAR_NAIVE = GHZConfiguration(
    state_generation_routine="naive",
    custom_qubits_array=[star_layout],
    shots=shots,
    fidelity_routine="coherences",
    rem=False,
    use_dd=False,
    max_circuits_per_batch=100,
)

benchmark_star_naive = GHZBenchmark(star_backend, STAR_NAIVE)
run_star_naive = benchmark_star_naive.run()
fidelity_star_naive = ghz_fidelity(benchmark_star_naive.analyze(), star_layout)

print(f"naive (chain): {fidelity_star_naive:.4f}")

In [ ]:
## The fan-out, which is the shape the chip actually has. Only the routine differs.
STAR_FANOUT = GHZConfiguration(
    state_generation_routine="star",
    custom_qubits_array=[star_layout],
    shots=shots,
    fidelity_routine="coherences",
    rem=False,
    use_dd=False,
    max_circuits_per_batch=100,
)

benchmark_star_fanout = GHZBenchmark(star_backend, STAR_FANOUT)
run_star_fanout = benchmark_star_fanout.run()
fidelity_star_fanout = ghz_fidelity(benchmark_star_fanout.analyze(), star_layout)

print(f"star (fan-out): {fidelity_star_fanout:.4f}")

In [ ]:
## The circuits the device actually executed, after MOVE routing.
group_name = f"{star_layout}_native_ghz"
circuit_naive = run_star_naive.circuits["transpiled_circuits"][group_name].circuits[0]
circuit_fanout = run_star_fanout.circuits["transpiled_circuits"][group_name].circuits[0]

ops_naive = dict(circuit_naive.count_ops())
ops_fanout = dict(circuit_fanout.count_ops())

print(f"{'':<16}{'naive':>8}{'star':>8}{'diff':>8}")
for gate in ("move", "cz"):
    a, b = ops_naive.get(gate, 0), ops_fanout.get(gate, 0)
    print(f"{gate:<16}{a:>8}{b:>8}{b - a:>+8}")
print(f"{'depth':<16}{circuit_naive.depth():>8}{circuit_fanout.depth():>8}{circuit_fanout.depth() - circuit_naive.depth():>+8}")
print(f"{'fidelity':<16}{fidelity_star_naive:>8.4f}{fidelity_star_fanout:>8.4f}{fidelity_star_fanout - fidelity_star_naive:>+8.4f}")

print(f"\nnaive ops: {ops_naive}")
print(f"star  ops: {ops_fanout}")

In [ ]:
circuit_naive.draw("mpl", fold=True, idle_wires=False)

In [ ]:
circuit_fanout.draw("mpl", fold=True, idle_wires=False)

## Recap

- Every benchmark in ``iqm-benchmarks`` is the same three steps: build a **configuration**, ``run()`` it on
  hardware, ``analyze()`` the counts into **observations** and **plots**. Analysis is free to repeat; only
  the run costs device time.
- A configuration is data, not a call: it can be copied, diffed and stored next to its results, which is
  what makes a controlled comparison between two settings easy to set up.
- ``custom_qubits_array`` takes **many layouts at once**, so one run and one analysis produced the whole
  5-to-20-qubit curve rather than sixteen separate experiments to bookkeep.
- The fidelity comes from the **multiple quantum coherences** sweep: a fixed $2n + 2$ phase points plus one
  populations circuit, so the cost follows the qubit count and the estimate arrives without an error bar.
- The GHZ benchmark estimates the state fidelity, and **0.5 is the line that matters**: above it, the state
  is certified genuinely multipartite entangled. Sweeping the size is what tells you where that line is
  crossed on this device today.

### The moral: state preparation is an error reduction technique

Which circuit you use to build the state is a free parameter, and it is not free of consequences. It costs
no extra shots, no extra calibration and no extra hardware time - but it decides how deep the circuit is
and which gates carry the entanglement.

- **``"tree"`` is the right default on a crystal**, for two reasons that compound: it routes the
  entanglement along a *minimum spanning tree weighted by CZ error*, so the good gates carry it, and it
  grows the state outward from the most central qubit, so the same $n-1$ CZs pack into logarithmically
  many **layers** instead of linearly many. The depth table in section 6 is that second claim, measured.
- **``"naive"``, the textbook chain, is blind on both counts**: linear depth, and no idea which gates on
  the chip are good.
- **On a star device the ranking changes entirely**, because the cheap circuit is a different shape: a
  fan-out from a single hub matches a chip whose qubits all meet at one resonator, while the chain has to
  be shuttled through that resonator link by link. Section 7 runs both and puts the `MOVE` counts and the
  fidelities side by side - that comparison is the whole argument.
- The general rule behind all of it: **match the circuit to the topology first, then to today's
  calibration.** A routine that ignores either is leaving fidelity on the table.

[4_ghz_error_reduction.ipynb](4_ghz_error_reduction.ipynb) starts from the tree circuit and adds the
techniques that cost something - dynamical decoupling, qubit selection, readout mitigation - one at a time.